# NB-000 — Stack Probe

Closes Step 6. Proves the free stack is wired correctly **before any build**.
Three live calls (Gemini generation, Groq generation, Judge JSON), each asserted.
Live-call cells are tagged `integration` in cell metadata so the default gate run can
skip them — see `doc/DECISIONS.md` D9 and `doc/notes/00b_probe_notes.md`.

NB-000 carries **no package code** — nothing here is promoted; the config package lands
in Mod 2 (`doc/task/02`).

## Setup — repo root + env

In [1]:
# Repo root => makes the relative "data/..." paths work no matter where Jupyter launched.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
load_dotenv(ROOT / ".env", override=True)

print("repo root:", ROOT)

repo root: /home/dipak/agentic/step9_llmops


In [2]:
os.environ["LANGCHAIN_PROJECT"] = os.environ.get("LANGCHAIN_PROJECT", "step9_llmops")
os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "NA")
# FIX: default bumped from gemini-3.5-flash -> gemini-3.6-flash.
# 3.5-flash 404'd mid-run (see doc/notes/00b_probe_notes.md); Google's own error
# pointed at gemini-3.6-flash as the replacement. Using `or` (not just `.get(..., default)`)
# so an *empty-string* env var also falls back to the default, not just a missing one.
os.environ["GEMINI_MODEL"] = os.environ.get("GEMINI_MODEL") or "gemini-3.5-flash"
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "NA")
os.environ["GROQ_MODEL"] = os.environ.get("GROQ_MODEL") or "openai/gpt-oss-120b"

import warnings
warnings.filterwarnings("ignore", message="Direct use of automatic function calling")

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
import json
from langchain_core.messages import HumanMessage

print("GEMINI_MODEL:", os.environ["GEMINI_MODEL"])
print("GROQ_MODEL  :", os.environ["GROQ_MODEL"])

GEMINI_MODEL: gemini-3.5-flash
GROQ_MODEL  : openai/gpt-oss-120b


In [3]:
USER_PROMPT = "In one sentence, what is RAG?"

gemini_llm = ChatGoogleGenerativeAI(
    model=os.environ["GEMINI_MODEL"],
    temperature=0,
    google_api_key=os.environ.get("GEMINI_API_KEY"),
)

In [4]:
groq_llm = ChatGroq(
    model=os.environ["GROQ_MODEL"],
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"],
)

In [5]:
def extract_text(response) -> str:
    """Return plain text regardless of whether .content is a str or a list of
    content blocks (Gemini 3.x thinking models return blocks; Groq returns str)."""
    content = response.content
    if isinstance(content, str):
        return content.strip()
    return "".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    ).strip()

In [6]:
from openai import OpenAI

judge_client = OpenAI(
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
)

In [7]:
# Fail fast, before spending a live call, if keys were never loaded.
for _key in ("GEMINI_API_KEY", "GROQ_API_KEY"):
    assert os.environ.get(_key, "NA") != "NA", f"{_key} not set — check .env"
print("API keys present.")

API keys present.


## 1. Gemini generation — live call

In [8]:
gemini_response = gemini_llm.invoke([HumanMessage(content=USER_PROMPT)])
gemini_text = extract_text(gemini_response)

print("\n=== Gemini ===")
print(gemini_text)

# --- assert ---
assert isinstance(gemini_text, str), f"Expected str, got {type(gemini_text)}"
assert len(gemini_text) > 0, "Gemini returned empty content"
print("[PASS] Gemini call returned non-empty text.")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



=== Gemini ===
**Retrieval-Augmented Generation (RAG)** is an AI technique that enhances the accuracy and reliability of large language models by fetching relevant information from external knowledge bases before generating a response.
[PASS] Gemini call returned non-empty text.


In [9]:
print(f"Gemini chars: {len(gemini_text)} | words: {len(gemini_text.split())}")

Gemini chars: 220 | words: 29


## 2. Groq generation — live call

In [10]:
groq_response = groq_llm.invoke([HumanMessage(content=USER_PROMPT)])
groq_text = extract_text(groq_response)

print("=== Groq ===")
print(groq_text)

# --- assert ---
assert isinstance(groq_text, str), f"Expected str, got {type(groq_text)}"
assert len(groq_text) > 0, "Groq returned empty content"
print("[PASS] Groq call returned non-empty text.")

=== Groq ===
Retrieval‑Augmented Generation (RAG) is a hybrid AI approach that combines a language model’s generative capabilities with a real‑time search of external documents or databases, retrieving relevant information to ground and improve the model’s output.
[PASS] Groq call returned non-empty text.


## 3. Judge JSON — live call

In [11]:
JUDGE_PROMPT = f"""You are an evaluation judge. Question: "{USER_PROMPT}"

Answer A (Groq): {groq_text}
Answer B (Gemini): {gemini_text}

Return ONLY valid JSON, no markdown fences, no prose, exactly this schema:
{{
  "winner": "A" | "B" | "tie",
  "reasoning": "<one sentence>",
  "score_a": <int 1-10>,
  "score_b": <int 1-10>
}}"""

In [12]:
print(JUDGE_PROMPT)

You are an evaluation judge. Question: "In one sentence, what is RAG?"

Answer A (Groq): Retrieval‑Augmented Generation (RAG) is a hybrid AI approach that combines a language model’s generative capabilities with a real‑time search of external documents or databases, retrieving relevant information to ground and improve the model’s output.
Answer B (Gemini): **Retrieval-Augmented Generation (RAG)** is an AI technique that enhances the accuracy and reliability of large language models by fetching relevant information from external knowledge bases before generating a response.

Return ONLY valid JSON, no markdown fences, no prose, exactly this schema:
{
  "winner": "A" | "B" | "tie",
  "reasoning": "<one sentence>",
  "score_a": <int 1-10>,
  "score_b": <int 1-10>
}


In [13]:
EXPECTED_KEYS = {"winner", "reasoning", "score_a", "score_b"}

def strip_fences(raw: str) -> str:
    """Some models wrap JSON in ```json ... ``` even when told not to."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").removeprefix("json").strip()
    return raw

In [14]:
judge_response = judge_client.chat.completions.create(
    model=os.environ["LLM_MODEL"],
    messages=[{"role": "user", "content": JUDGE_PROMPT}],
    temperature=0,
)

judge_raw = strip_fences(judge_response.choices[0].message.content)

try:
    judge_json = json.loads(judge_raw)
except json.JSONDecodeError as e:
    raise AssertionError(f"Judge did not return valid JSON: {e}\nRaw: {judge_raw!r}") from e

# --- schema asserts ---
missing = EXPECTED_KEYS - judge_json.keys()
assert not missing, f"Missing keys in judge output: {missing}"
assert judge_json["winner"] in ("A", "B", "tie"), f"Bad winner: {judge_json['winner']}"
assert isinstance(judge_json["score_a"], int) and 1 <= judge_json["score_a"] <= 10
assert isinstance(judge_json["score_b"], int) and 1 <= judge_json["score_b"] <= 10

print("\n=== Judge ===")
print(json.dumps(judge_json, indent=2))
print("[PASS] Judge returned valid JSON matching expected schema.")


=== Judge ===
{
  "winner": "B",
  "reasoning": "More concise and directly addresses the question.",
  "score_a": 8,
  "score_b": 9
}
[PASS] Judge returned valid JSON matching expected schema.


## Stack probe result

In [15]:
print("=" * 50)
print("NB-000 STACK PROBE: ALL THREE CALLS PASSED")
print("=" * 50)
print(f"  Gemini : {os.environ['GEMINI_MODEL']}")
print(f"  Groq   : {os.environ['GROQ_MODEL']}")
print(f"  Judge  : {os.environ['LLM_MODEL']}")
print("Configuration probe completed.")

NB-000 STACK PROBE: ALL THREE CALLS PASSED
  Gemini : gemini-3.5-flash
  Groq   : openai/gpt-oss-120b
  Judge  : Qwen/Qwen2.5-7B-Instruct-AWQ
Configuration probe completed.
